# Retrievers (RAG)

From the [Retrievers documentation](../../../docs/source/workflows/retrievers.md):

> *"Retrievers are used to retrieve relevant documents from a vector database."*

## Supported Retriever Providers

| Provider | Type | Description |
|----------|------|-------------|
| **NVIDIA NIM** | `nemo_retriever` | NVIDIA Inference Microservice (NIM) |
| **Milvus** | `milvus_retriever` | Open-source vector database |

## Supported Embedder Providers

From the [Embedders documentation](../../../docs/source/workflows/embedders.md):

| Provider | Type | Description |
|----------|------|-------------|
| **NVIDIA NIM** | `nim` | NVIDIA Inference Microservice (NIM) |
| **OpenAI** | `openai` | OpenAI embedding API |
| **Azure OpenAI** | `azure_openai` | Azure OpenAI embedding API |

## What You'll Learn

1. Setting up Milvus and bootstrapping data
2. Embedder configuration
3. Milvus retriever configuration
4. Using retrievers as functions for agents

## Prerequisites

Before running this notebook, from the **NeMo Agent Toolkit repository root**:

1. **Start Milvus** with Docker:
```bash
docker compose -f examples/deploy/docker-compose.milvus.yml up -d
```

2. **Bootstrap data** into Milvus collections:
```bash
source .venv/bin/activate
./scripts/bootstrap_milvus.sh
```

This creates two collections:
- `cuda_docs` - CUDA documentation
- `mcp_docs` - Model Context Protocol documentation

For more details, see the [Simple RAG Example README](../../RAG/simple_rag/README.md).


In [ ]:
import getpass
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Load environment variables from .env file
load_dotenv()

# Check for NVIDIA API key (required for NIM embedder)
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - examples may fail")


## 1. Embedder Configuration

Embedders convert text to vectors for similarity search:


In [ ]:
from nat.embedder.nim_embedder import NIMEmbedder

# NIM Embedder for NVIDIA embedding models
embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",  # NVIDIA embedding model
    truncate="END",                         # How to handle long text
    name="nim_embedder",
)

print(f"✅ Embedder: {embedder.model_name}")


## 2. Milvus Retriever

Milvus is a popular open-source vector database for RAG. We'll connect to the `cuda_docs` collection (created during bootstrap):


In [ ]:
from pydantic import HttpUrl

from nat.retriever.milvus.register import MilvusRetriever

# Milvus retriever for CUDA documentation
cuda_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),  # Milvus server URL
    collection_name="cuda_docs",            # Collection created by bootstrap script
    embedder=embedder,                      # Embedder for vectorizing queries
    top_k=10,                               # Number of results to return
    name="cuda_retriever",
)
print(f"✅ CUDA Retriever: {cuda_retriever.collection_name}")

# Milvus retriever for MCP documentation
mcp_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="mcp_docs",
    embedder=embedder,
    top_k=10,
    name="mcp_retriever",
)
print(f"✅ MCP Retriever: {mcp_retriever.collection_name}")


## 3. Retriever Tool

Use the retriever as a tool for agents:


In [ ]:
from nat.tool.retriever import NatRetrieverTool

# Create retriever tools for the agent
cuda_retriever_tool = NatRetrieverTool(
    nat_retriever=cuda_retriever,  # Note: use nat_retriever parameter
    topic="Retrieve documentation for NVIDIA's CUDA library",
    name="cuda_retriever_tool",
)
print(f"✅ CUDA Tool: {cuda_retriever_tool.name}")

mcp_retriever_tool = NatRetrieverTool(
    nat_retriever=mcp_retriever,
    topic="Retrieve information about Model Context Protocol (MCP)",
    name="mcp_retriever_tool",
)
print(f"✅ MCP Tool: {mcp_retriever_tool.name}")


## 4. RAG Agent Example


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=4096,
    name="nim_llm",
)

# RAG agent with multiple retriever tools
rag_agent = NatReActAgent(
    tools=[cuda_retriever_tool, mcp_retriever_tool],
    llm=llm,
    verbose=True,
    additional_instructions=(
        "Use the retriever tools to search for relevant information before answering. "
        "Use cuda_retriever_tool for CUDA-related questions and mcp_retriever_tool for MCP questions."
    ),
)

workflow = NatWorkflow(entrypoint=rag_agent)
print(f"✅ RAG Agent created with {len(rag_agent.tools)} retriever tools")


### Test the RAG Agent

Let's query the agent about CUDA installation:


In [ ]:
# Test with a CUDA question
result = await workflow.prompt("How do I install CUDA on Linux?")
print(f"🤖 Response:\n{result}")


### Save Configuration


In [ ]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "rag_agent.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Saved to: {config_path}")


## CLI Commands

```bash
# Run the RAG agent
nat run --config_file configs/rag_agent.yaml --input "How do I install CUDA?"

# Or use the pre-configured example
nat run --config_file examples/RAG/simple_rag/configs/milvus_rag_config.yml \
    --input "What is CUDA?"
```

## Bootstrapping Custom Data

To add your own documents to Milvus, use the `langchain_web_ingest.py` script:

```bash
python scripts/langchain_web_ingest.py \
    --urls https://your-docs-url.com/page1 \
    --urls https://your-docs-url.com/page2 \
    --collection_name your_collection \
    --milvus_uri http://localhost:19530
```

## Summary

✅ **Milvus setup** - Docker Compose deployment  
✅ **Data bootstrap** - Ingest documents into collections  
✅ **Embedders** - Convert text to vectors (`NIMEmbedder`)  
✅ **Retrievers** - Search vector databases (`MilvusRetriever`)  
✅ **Retriever tools** - `NatRetrieverTool` for agent integration  
✅ **RAG Agent** - Complete RAG workflow  

## Next Steps

- **[08_configuration_guide.ipynb](./08_configuration_guide.ipynb)** - YAML configuration
- **[09_evaluation.ipynb](./09_evaluation.ipynb)** - Evaluate agent quality
